# 14.15 只練1、3、5輪，可以在2、4輪交卷嗎？


可以把推論程式設成二輪或四輪，但是否答得好要測。這個問題最有意思的地方，是**可執行**和**可泛化**並不相同。

要做乾淨的機制實驗，訓練時只抽1、3、5作終止深度，每次只在末端算 loss；不要在中間所有輪都計分。訓練到五輪時，第二、四輪仍然是通往第五輪的內部狀態，也收到間接梯度，所以應稱它們為「**未直接監督的終止深度**」。不能說完全沒有見過第二、四輪的狀態。

凍結模型後，在相同留出題上測1、2、3、4、5、6輪。二、四在訓練終止深度範圍內，叫深度**插值**；六超出最大終止深度，叫**外推**。這些詞標示測試位置，沒有承諾連續平滑。固定三輪訓練也要另列基線，才能看出變動終止配方改了什麼。

下面是反例：用手工更新每輪把狀態變號，正確類別總是正號。它沒有學習，也不是 Transformer 效果，只證明「所有奇數輪好」在邏輯上推不出「偶數輪也好」。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
state = -1
for r in range(1, 7):
    state = -state
    probability_correct = 0.9 if state > 0 else 0.1
    print(r, probability_correct)

奇數輪正解機率0.9，偶數輪0.1。共享更新可以產生這種週期，因此不能只憑共享權重或奇數輪訓練來保證偶數表現。實際網路未必這樣振盪，但要用完整曲線排除，而不是只挑表現最好的幾個點。

本課的最小兩配方就是固定3輪末端 CE，與抽1／3／5輪末端 CE。資料、起點、更新或工作量對照方式都要記下，測試各輪的 loss、答案正確率、時間與記憶體。正式產品可以另外監督所有要部署的輪數；那是為可用性做的工程選擇，不再拿來當奇數輪插值證據。

練習：如果加入第2輪監督而仍不監督第4輪，該如何標示？要完整寫監督集合與抽樣規則，第2輪不再是未直接監督；不要只用「跳躍式訓練」一個名稱掩蓋差異。
